In [1]:
import sqlite3
import random
import time
import requests
import pandas as pd

random.seed(42)

Part 1.1: Create the database

In [2]:
conn = sqlite3.connect("school.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS enrollments")
cur.execute("DROP TABLE IF EXISTS students")

cur.execute("""
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name TEXT, major TEXT, gpa REAL
)""")

cur.execute("""
CREATE TABLE enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER, course TEXT, grade TEXT,
    FOREIGN KEY (student_id) REFERENCES students(student_id)
)""")

students = [
    (1, "Alice", "CS", 3.9), (2, "Bob", "CS", 3.6),
    (3, "Chaeyoung", "CS", 3.2), (4, "Daniel", "CS", 3.8),
    (5, "Eunji", "Math", 3.7), (6, "Farid", "Math", 3.1),
    (7, "Grace", "Math", 3.4), (8, "Hana", "Biology", 3.5),
    (9, "Ivan", "Biology", 2.9), (10, "Jisoo", "Physics", 3.3),
]
cur.executemany("INSERT INTO students VALUES (?, ?, ?, ?)", students)

enrollments = [
    (1, 1, "Data Science", "A"), (2, 1, "Algorithms", "A"),
    (3, 2, "Data Science", "B"), (4, 3, "Databases", "B"),
    (5, 4, "Algorithms", "A"), (6, 5, "Linear Algebra", "A"),
    (7, 6, "Linear Algebra", "C"), (8, 7, "Statistics", "B"),
    (9, 8, "Genetics", "B"), (10, 9, "Genetics", "C"),
]
cur.executemany("INSERT INTO enrollments VALUES (?, ?, ?, ?)", enrollments)

conn.commit()

Part 1.2: SELECT

In [3]:
print(pd.read_sql("SELECT * FROM students;", conn))

query = """
    SELECT name, gpa
    FROM students
    WHERE major = 'CS' AND gpa >= 3.5
    ORDER BY gpa DESC
"""
print(pd.read_sql(query, conn))

   student_id       name    major  gpa
0           1      Alice       CS  3.9
1           2        Bob       CS  3.6
2           3  Chaeyoung       CS  3.2
3           4     Daniel       CS  3.8
4           5      Eunji     Math  3.7
5           6      Farid     Math  3.1
6           7      Grace     Math  3.4
7           8       Hana  Biology  3.5
8           9       Ivan  Biology  2.9
9          10      Jisoo  Physics  3.3
     name  gpa
0   Alice  3.9
1  Daniel  3.8
2     Bob  3.6


Part 1.3: JOIN

In [4]:
query = """
    SELECT students.name, enrollments.course, enrollments.grade
    FROM students
    INNER JOIN enrollments ON students.student_id = enrollments.student_id
"""
print(pd.read_sql(query, conn))

# LEFT JOIN keeps students with no enrollment (Jisoo shows NaN)
query = """
    SELECT students.name, enrollments.course, enrollments.grade
    FROM students
    LEFT JOIN enrollments ON students.student_id = enrollments.student_id
"""
print(pd.read_sql(query, conn))

# Same thing in Pandas
students_df = pd.read_sql("SELECT * FROM students", conn)
enroll_df = pd.read_sql("SELECT * FROM enrollments", conn)
print(pd.merge(students_df, enroll_df, on="student_id", how="left")[["name", "course", "grade"]])

        name          course grade
0      Alice    Data Science     A
1      Alice      Algorithms     A
2        Bob    Data Science     B
3  Chaeyoung       Databases     B
4     Daniel      Algorithms     A
5      Eunji  Linear Algebra     A
6      Farid  Linear Algebra     C
7      Grace      Statistics     B
8       Hana        Genetics     B
9       Ivan        Genetics     C
         name          course grade
0       Alice      Algorithms     A
1       Alice    Data Science     A
2         Bob    Data Science     B
3   Chaeyoung       Databases     B
4      Daniel      Algorithms     A
5       Eunji  Linear Algebra     A
6       Farid  Linear Algebra     C
7       Grace      Statistics     B
8        Hana        Genetics     B
9        Ivan        Genetics     C
10      Jisoo            None  None
         name          course grade
0       Alice    Data Science     A
1       Alice      Algorithms     A
2         Bob    Data Science     B
3   Chaeyoung       Databases     B
4  

Part 1.4: GROUP BY

In [5]:
query = """
    SELECT major, COUNT(*) AS n_students, AVG(gpa) AS avg_gpa
    FROM students
    GROUP BY major
    HAVING COUNT(*) > 2
    ORDER BY avg_gpa DESC
"""
print(pd.read_sql(query, conn))

# Same thing in Pandas
print(
    students_df.groupby("major")
    .agg(n_students=("gpa", "count"), avg_gpa=("gpa", "mean"))
    .query("n_students > 2")
    .sort_values("avg_gpa", ascending=False)
)

  major  n_students  avg_gpa
0    CS           4    3.625
1  Math           3    3.400
       n_students  avg_gpa
major                     
CS              4    3.625
Math            3    3.400


Part 2.1: Safe queries with ? placeholders

In [6]:
# UNSAFE: never do this
unsafe_name = "CS' OR '1'='1"
unsafe_query = f"SELECT name, gpa FROM students WHERE major = '{unsafe_name}'"
print(unsafe_query)   # WHERE clause becomes always true

# SAFE
cur.execute("SELECT name, gpa FROM students WHERE major = ?", ("CS",))
rows = cur.fetchall()
print(rows)

# The malicious string is treated as plain data, so it matches nothing
cur.execute("SELECT name, gpa FROM students WHERE major = ?", (unsafe_name,))
print(cur.fetchall())   # []

SELECT name, gpa FROM students WHERE major = 'CS' OR '1'='1'
[('Alice', 3.9), ('Bob', 3.6), ('Chaeyoung', 3.2), ('Daniel', 3.8)]
[]


Part 2.2: pd.read_sql()

In [7]:
query = """
    SELECT major, AVG(gpa) AS avg_gpa
    FROM students
    GROUP BY major
"""
df = pd.read_sql(query, conn)
print(df)

conn.close()

     major  avg_gpa
0  Biology    3.200
1       CS    3.625
2     Math    3.400
3  Physics    3.300


Part 3.1: REST API (Seoul weather)

In [8]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 37.5665,
    "longitude": 126.9780,
    "current_weather": True,
}

try:
    response = requests.get(url, params=params, timeout=10)
    print("status code:", response.status_code)   # 200 = success
    response.raise_for_status()
    data = response.json()
    print(data["current_weather"])
except requests.exceptions.RequestException:
    # fallback if there is no Wi-Fi
    data = {"current_weather": {"temperature": 18.4, "windspeed": 9.1}}

status code: 200
{'time': '2026-10-09T15:15', 'interval': 900, 'temperature': 16.1, 'windspeed': 1.0, 'winddirection': 315, 'is_day': 0, 'weathercode': 0}


Part 3.2: JSON to DataFrame

In [9]:
weather_df = pd.DataFrame([data["current_weather"]])
print(weather_df)

               time  interval  temperature  windspeed  winddirection  is_day  \
0  2026-10-09T15:15       900         16.1        1.0            315       0   

   weathercode  
0            0  


Part 3.3: Several cities with rate-limit pacing

In [10]:
cities = {
    "Seoul": (37.5665, 126.9780),
    "Busan": (35.1796, 129.0756),
    "Tokyo": (35.6762, 139.6503),
}

records = []
for city, (lat, lon) in cities.items():
    try:
        r = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={"latitude": lat, "longitude": lon, "current_weather": True},
            timeout=10,
        )
        r.raise_for_status()
        rec = r.json()["current_weather"]
        rec["city"] = city
        records.append(rec)
    except requests.exceptions.RequestException as e:
        print(f"{city}: request failed ({e})")
    time.sleep(1)   # pace your requests

cities_df = pd.DataFrame(records)
print(cities_df)

               time  interval  temperature  windspeed  winddirection  is_day  \
0  2026-10-09T15:15       900         16.1        1.0            315       0   
1  2026-10-09T15:15       900         19.6        4.5             61       0   
2  2026-10-09T15:15       900         16.9        3.3            347       0   

   weathercode   city  
0            0  Seoul  
1            1  Busan  
2            0  Tokyo  


In [11]:
import os
api_key = os.environ.get("MY_API_KEY")

Part 4: Ethical web scraping

In [12]:
from bs4 import BeautifulSoup

url = "http://books.toscrape.com"
headers = {"User-Agent": "KDU-DataScience-Student-Demo"}

response = requests.get(url, headers=headers, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")

titles = [h.get_text(strip=True) for h in soup.select("h3 a")]
print(titles[:5])

time.sleep(1)   # be polite before any next request

['A Light in the ...', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History ...']


Lab outcome: all three sources in DataFrames

In [14]:
# 1) SQL
conn = sqlite3.connect("school.db")
sql_df = pd.read_sql("SELECT major, AVG(gpa) AS avg_gpa FROM students GROUP BY major", conn)
conn.close()

# 2) API
api_df = cities_df if len(cities_df) else pd.DataFrame([data["current_weather"]])

# 3) Web
web_df = pd.DataFrame({"book_title": titles})

for name, d in [("SQL", sql_df), ("API", api_df), ("Web", web_df)]:
    print(f"--- {name} ---")
    print(d.head(), "\n")

--- SQL ---
     major  avg_gpa
0  Biology    3.200
1       CS    3.625
2     Math    3.400
3  Physics    3.300 

--- API ---
               time  interval  temperature  windspeed  winddirection  is_day  \
0  2026-10-09T15:15       900         16.1        1.0            315       0   
1  2026-10-09T15:15       900         19.6        4.5             61       0   
2  2026-10-09T15:15       900         16.9        3.3            347       0   

   weathercode   city  
0            0  Seoul  
1            1  Busan  
2            0  Tokyo   

--- Web ---
                     book_title
0            A Light in the ...
1            Tipping the Velvet
2                    Soumission
3                 Sharp Objects
4  Sapiens: A Brief History ... 

